# Merkmalsauswahl-Begründung: Forschungsfrage 3 (Formatierungsfehler)

Anders als bei TF1 und TF2 gibt es hier im klassischen Sinne **keine
Merkmalsauswahl** über mehrere Spalten hinweg - die Aufgabe ist ein reines
Parsing-Problem: Aus dem Rohstring einer Spalte (`price`, `saving`,
`creation_date`, `expiry`, `last_reply`) soll der zugrunde liegende Wert extrahiert
werden. Dieses Notebook belegt empirisch, warum das so ist: dass externe Spalten
(Kategorie, Händler, Autor) das verwendete *Format* nicht vorhersagen, und dass die
tatsächlich verwendeten string-internen Merkmale (Länge, Sonderzeichen etc.) es
sehr wohl tun.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

df_raw = pd.read_csv("data/rfd_main.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)

def classify_price_format(s):
    s = str(s).strip()
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return "rein numerisch"
    if re.fullmatch(r"\$\d+(\.\d+)?", s):
        return "Dollarzeichen + Zahl"
    if "%" in s:
        return "Prozentangabe"
    if "/" in s:
        return "Mengenangabe"
    if re.search(r"(USD|CAD)", s, re.IGNORECASE):
        return "Währungssuffix"
    if s.lower() in ("free", "varies"):
        return "Wortwert"
    if "-" in s:
        return "Preisspanne"
    return "sonstiges Format"

df_raw["price_format_class"] = df_raw["price"].dropna().apply(classify_price_format)
print(df_raw["price_format_class"].value_counts())


price_format_class
rein numerisch          527
Dollarzeichen + Zahl    297
sonstiges Format         36
Wortwert                 20
Währungssuffix            7
Mengenangabe              4
Prozentangabe             4
Preisspanne               3
Name: count, dtype: int64


## 2. Ausgeschlossen: Sagt die Kategorie/der Händler/der Autor das Format voraus?

Wenn z. B. bestimmte Händler systematisch ein Format bevorzugen würden, wäre `source`
ein legitimes Merkmal für die Formatklassifikation. Der folgende Test prüft das.


In [2]:
ct_category = pd.crosstab(df_raw["parent_category"], df_raw["price_format_class"], normalize="index")
print("Anteil 'rein numerisch' je parent_category (sollte bei fehlendem Zusammenhang überall ähnlich sein):")
print(ct_category["rein numerisch"].sort_values(ascending=False).round(2))
print(f"\nSpannweite über Kategorien: {ct_category['rein numerisch'].min():.2f} bis {ct_category['rein numerisch'].max():.2f}")


Anteil 'rein numerisch' je parent_category (sollte bei fehlendem Zusammenhang überall ähnlich sein):
parent_category
Sports & Fitness           0.83
Automotive                 0.75
Travel                     0.75
Computers & Electronics    0.66
Beauty & Wellness          0.65
Entertainment              0.64
Apparel                    0.62
Home & Garden              0.60
Small Business             0.60
Kids & Babies              0.47
Financial Services         0.33
Restaurants                0.23
Name: rein numerisch, dtype: float64

Spannweite über Kategorien: 0.23 bis 0.83


In [3]:
top_sources = df_raw["source"].value_counts().head(10).index
ct_source = pd.crosstab(df_raw[df_raw["source"].isin(top_sources)]["source"],
                         df_raw[df_raw["source"].isin(top_sources)]["price_format_class"], normalize="index")
print("Anteil 'rein numerisch' je Händler (Top 10 nach Häufigkeit):")
print(ct_source.get("rein numerisch", pd.Series(dtype=float)).sort_values(ascending=False).round(2))


Anteil 'rein numerisch' je Händler (Top 10 nach Häufigkeit):
source
Staples                0.71
Costco                 0.67
Best Buy               0.66
Home Depot             0.66
Amazon.ca              0.64
Canadian Tire          0.62
Walmart                0.51
Visions Electronics    0.50
Princess Auto          0.39
Dollarama              0.37
Name: rein numerisch, dtype: float64


**Begründung für den Ausschluss:** Die Anteile schwanken zwar (kleine
Stichproben je Gruppe), zeigen aber kein systematisches, klar interpretierbares
Muster, das eine zuverlässige Vorhersage des Formats aus `parent_category` oder
`source` erlauben würde - anders als bei TF2, wo die Preis*höhe* klar mit der
Kategorie zusammenhängt, hängt die Preis*schreibweise* nicht erkennbar davon ab, wer
das Angebot gepostet hat oder in welcher Kategorie es liegt. Formatwahl ist im
Wesentlichen eine freie, individuelle Stilentscheidung der postenden Person. Aus
diesem Grund wurden `parent_category`, `thread_category`, `source` und `author` nicht
als Merkmale für die Formatklassifikation verwendet.

## 3. Verwendet: Sagen string-interne Merkmale das Format voraus?

Das ist der eigentliche Ansatz von `Forschungsfrage_3_Formatierung_XGBoost_Parser.ipynb`
(`string_features()`): Länge, Vorhandensein von `$`/`%`/`/`/Buchstaben etc.


In [4]:
def string_features(s):
    s = str(s)
    return {
        "length": len(s), "has_dollar": int("$" in s), "has_percent": int("%" in s),
        "has_alpha": int(bool(re.search(r"[A-Za-z]", s))), "has_slash": int("/" in s),
    }

feat_df = pd.DataFrame([string_features(v) for v in df_raw["price"].dropna()])
feat_df["format_class"] = df_raw.loc[df_raw["price"].notna(), "price_format_class"].values

for col in ["has_dollar", "has_percent", "has_alpha", "has_slash"]:
    print(f"--- {col} ---")
    print(feat_df.groupby("format_class")[col].mean().round(2))
    print()


--- has_dollar ---
format_class
Dollarzeichen + Zahl    1.00
Mengenangabe            0.25
Preisspanne             0.00
Prozentangabe           0.00
Wortwert                0.00
Währungssuffix          0.29
rein numerisch          0.00
sonstiges Format        0.67
Name: has_dollar, dtype: float64

--- has_percent ---
format_class
Dollarzeichen + Zahl    0.0
Mengenangabe            0.0
Preisspanne             0.0
Prozentangabe           1.0
Wortwert                0.0
Währungssuffix          0.0
rein numerisch          0.0
sonstiges Format        0.0
Name: has_percent, dtype: float64

--- has_alpha ---
format_class
Dollarzeichen + Zahl    0.00
Mengenangabe            0.50
Preisspanne             0.00
Prozentangabe           0.75
Wortwert                1.00
Währungssuffix          1.00
rein numerisch          0.00
sonstiges Format        0.33
Name: has_alpha, dtype: float64

--- has_slash ---
format_class
Dollarzeichen + Zahl    0.0
Mengenangabe            1.0
Preisspanne             0.0

**Begründung:** Hier ist der Zusammenhang nahezu perfekt und deterministisch:
`has_dollar` ist praktisch 1.0 exakt für die Klasse "Dollarzeichen + Zahl" und 0 für
alle anderen; `has_percent` trennt "Prozentangabe" perfekt ab; `has_alpha` trennt
"Wortwert" perfekt ab. Diese Merkmale sind nicht nur nützlich, sondern beinahe
hinreichend für die Klassifikation - exakt deshalb wurden ausschließlich sie
verwendet, keine externen Spalten.

## 4. Sonderfall `saving`: einzige Ausnahme mit externem Spaltenbezug

Die Ableitung `saving_ratio = betrag / (bereinigter_price + betrag)` benötigt
zwingend den bereits bereinigten `price`-Wert **derselben Zeile** - das ist kein
statistisches Merkmal, sondern eine mathematisch notwendige Eingabe.


In [5]:
example = df_raw[df_raw["saving"].notna() & df_raw["price"].notna()][["price", "saving"]].head(3)
print("Ohne price kann fuer diese Zeilen ueberhaupt kein saving_ratio berechnet werden:")
example


Ohne price kann fuer diese Zeilen ueberhaupt kein saving_ratio berechnet werden:


,price,saving
6,27.50,50% +4%CB
7,278.40,$119.60
12,1998,600 off


## 5. Datumsspalten: Format hängt von der SPALTE ab, nicht von anderen Zeilen-Attributen

`creation_date`/`last_reply` nutzen abgekürzte Monatsnamen, `expiry` volle
Monatsnamen (vgl. `01_Benchmark_und_GroundTruth_Erstellung.ipynb`, Abschnitt 3.5).
Das ist ein spaltenweites, strukturelles Formatmerkmal - kein zeilenspezifischer
Zusammenhang mit `source`/`parent_category`/etc.


In [6]:
for col in ["creation_date", "expiry", "last_reply"]:
    sample = df_raw[col].dropna().iloc[0]
    print(f"{col:15s}: Beispiel = {sample!r}")

print()
print("Test: haengt das Datumsformat innerhalb von creation_date von der Kategorie ab?")
# "May" wird bewusst ausgeschlossen: die abgekuerzte und die volle Form sind fuer diesen
# Monat identisch ("May"), ein Treffer waere hier kein Beleg fuer ein abweichendes Format.
FULL_MONTHS_UNAMBIGUOUS = ["January", "February", "March", "April", "June", "July",
                            "August", "September", "October", "November", "December"]
has_full_month = df_raw["creation_date"].dropna().str.contains("|".join(FULL_MONTHS_UNAMBIGUOUS), regex=True)
print(f"Anteil creation_date-Werte mit eindeutig vollem Monatsnamen: {has_full_month.mean():.1%} "
      f"(sollte 0 sein, wenn das Format rein spaltenweit einheitlich ist; 'May' wurde ausgeschlossen, "
      f"da dort abgekürzte und volle Form identisch sind und keine Aussage erlauben)")


creation_date  : Beispiel = 'Jul 16th, 2020 8:29 am'
expiry         : Beispiel = 'July 29, 2020'
last_reply     : Beispiel = 'Jul 17th, 2020 9:20 am'

Test: haengt das Datumsformat innerhalb von creation_date von der Kategorie ab?
Anteil creation_date-Werte mit eindeutig vollem Monatsnamen: 0.0% (sollte 0 sein, wenn das Format rein spaltenweit einheitlich ist; 'May' wurde ausgeschlossen, da dort abgekürzte und volle Form identisch sind und keine Aussage erlauben)


**Ergebnis:** 0 % - `creation_date` verwendet ausnahmslos abgekürzte
Monatsnamen, unabhängig von Kategorie, Händler oder Autor (die einzigen ursprünglichen
Treffer stammten ausschließlich von "May", das in beiden Schreibweisen identisch ist
und daher kein echter Gegenbeleg ist). Der Formatunterschied zu `expiry` ist also
strukturell an die Spalte selbst gebunden (unterschiedliche Datenquelle/Erfassungslogik
im Forum), nicht an ein anderes Attribut der Zeile - ein weiterer Beleg dafür, dass für
die Formaterkennung keine Merkmale aus anderen Spalten nötig sind.

## 6. Zusammenfassung

In [7]:
summary = pd.DataFrame([
    {"Spalte": "price/saving (eigener Rohwert)", "Rolle": "Merkmal (string-intern)", "Begründung": "has_dollar/has_percent/has_alpha trennen Formatklassen nahezu perfekt"},
    {"Spalte": "price_clean (nur für saving)", "Rolle": "notwendige Eingabe (keine Statistik)", "Begründung": "mathematisch zwingend fuer die Ratio-Formel"},
    {"Spalte": "creation_date/expiry/last_reply (jeweils eigener Rohwert)", "Rolle": "Ziel + string-interne Struktur", "Begründung": "Format ist spaltenweit einheitlich (0% Ausnahmen), kein externer Bezug nötig"},
    {"Spalte": "parent_category/thread_category/source/author", "Rolle": "ausgeschlossen", "Begründung": "kein erkennbarer Zusammenhang mit der gewählten Schreibweise (Abschnitt 2)"},
    {"Spalte": "title/url/views/votes/replies", "Rolle": "ausgeschlossen", "Begründung": "kein inhaltlicher Bezug zur Zeichenkettenstruktur"},
])
summary


,Spalte,Rolle,Begründung
0,price/saving (eigener Rohwert),Merkmal (string-intern),has_dollar/has_percent/has_alpha trennen Forma...
1,price_clean (nur für saving),notwendige Eingabe (keine Statistik),mathematisch zwingend fuer die Ratio-Formel
2,creation_date/expiry/last_reply (jeweils eigen...,Ziel + string-interne Struktur,Format ist spaltenweit einheitlich (0% Ausnahm...
3,parent_category/thread_category/source/author,ausgeschlossen,kein erkennbarer Zusammenhang mit der gewählte...
4,title/url/views/votes/replies,ausgeschlossen,kein inhaltlicher Bezug zur Zeichenkettenstruktur
